# CoalGameRec — Round-6 Required Experiments (v19 response)

Runs the six experiment tracks released for the round-6 Discovery-AI reviews. Every cell streams
output live **and** persists logs + artifacts you can hand back. Everything is **resume-safe**:
interrupt and re-run; completed seeds/datasets are skipped or merged.

**Recommended order (by reviewer priority):**

| Cell | Experiment | Script | Est. time (M-series MPS) |
|---|---|---|---|
| 3 | **LOO λ-sweep + validation-tuned λ** (both datasets) | `run_loo_lambda_sweep.py` | ~1 h Amazon / ~3.5 h ML-1M (without Shapley curves) |
| 4 | **Second backbone: NGCF** (identical protocol) | `run_second_backbone.py --backbone ngcf` | ~2–3 h Amazon / ~12–16 h ML-1M (with Shapley) |
| 5 | Multi-seed design ablations (seeds 43+) | `run_design_ablations.py` | ~30–40 min/seed Amazon, ~10–12 h/seed ML-1M |
| 6 | Multi-seed masked-forward faithfulness (CPU) | `run_masked_forward_faithfulness.py` | ~30–40 min/seed/dataset |
| 7 | Attribution stability + model-randomization sanity | `run_randomization_sanity.py` | ~30 min Amazon / ~1.5 h ML-1M |
| 8 | Validation-negative sensitivity (50/100/500) | `run_negset_sensitivity.py` | ~1 h Amazon / ~2–3 h ML-1M |

**Notes**
- The corrected statistical tables (permutation p-values, Wilcoxon, d_z CIs, Friedman–Nemenyi,
  MDE/TOST power) were already recomputed in the sandbox from existing artifacts — no runs needed.
- Cell 4's NGCF is the structurally different backbone the reviewers demanded (nonlinear
  W1/W2 aggregation + LeakyReLU, mean layer readout documented in the config).
- After each long cell, you can run Cell 9 (status) and Cell 10 (push) immediately — partial
  artifacts are valid and the remaining cells resume cleanly.


In [ ]:
# Cell 1 — Setup: paths, device detection, streaming helper
from pathlib import Path
import os, sys, platform, subprocess

CWD = Path.cwd().resolve()
CODE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
RESULTS = CODE_DIR / "results" / "journal_runs"
LOGDIR = RESULTS / "_notebook_logs"
LOGDIR.mkdir(parents=True, exist_ok=True)

import torch
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

RUN_ENV = dict(os.environ, COALGAME_DEVICE=DEVICE)

def stream(cmd, env=None, log=None, cwd=None):
    """Run cmd, stream output live into the notebook AND tee it to a persistent log."""
    print("RUN:", " ".join(cmd))
    lf = open(log, "a") if log else None
    if lf: lf.write("\n$ " + " ".join(cmd) + "\n")
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          cwd=str(cwd or CODE_DIR), env=env or RUN_ENV, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="")
            if lf: lf.write(line); lf.flush()
    if lf: lf.close()
    print("exit code:", p.returncode)
    return p.returncode

print("CODE_DIR:", CODE_DIR)
print("device  :", DEVICE, "| torch", torch.__version__, "| python", platform.python_version())
print("platform:", platform.platform())


In [ ]:
# Cell 2 — Environment check (source runs, splits, scripts, NGCF class present?)
required = {
    "ML-1M v3 splits": RESULTS / "ml1m_lightgcn_v3_prospective" / "splits" / "train.parquet",
    "Amazon v3 splits": RESULTS / "amazon_books_lightgcn_v3_prospective" / "splits" / "train.parquet",
    "ML-1M item-vector report": RESULTS / "ml1m_lightgcn_v3_prospective" / "item_vectors_report.json",
    "Amazon item-vector report": RESULTS / "amazon_books_lightgcn_v3_prospective" / "item_vectors_report.json",
}
ok = True
for name, path in required.items():
    print(f"{'OK ' if path.exists() else 'MISSING'} {name}")
    ok = ok and path.exists()
for script in ["run_loo_lambda_sweep.py", "run_second_backbone.py", "run_negset_sensitivity.py",
               "run_randomization_sanity.py", "run_design_ablations.py", "run_masked_forward_faithfulness.py"]:
    p = CODE_DIR / "scripts" / script
    print(f"{'OK ' if p.exists() else 'MISSING'} script: {script}")
    ok = ok and p.exists()

sys.path.insert(0, str(CODE_DIR))
from coalgamerec.models import NGCF, LightGCN
print("OK  NGCF backbone class importable")
print("\nReady." if ok else "\nFix missing pieces before running experiments.")


In [ ]:
# Cell 3 — EXPERIMENT 1: LOO λ-sweep + validation-tuned λ (reviewers R1#6 / R2#4).
# Produces the missing LOO curves for λ ∈ {0, .05, .10, .20, .40} (5 seeds) and selects
# per-family λ on VALIDATION NDCG@20, reporting test NDCG@20 once (proper tuning protocol).
# WITH_SHAPLEY_SWEEP=True also re-emits Shapley curves from this re-execution (+~9 h on ML-1M);
# False keeps runtime short (Shapley curves already exist in the v3 sweep artifact).
import os

DATASETS = ["amazon", "ml1m"]          # amazon first (fast)
WITH_SHAPLEY_SWEEP = False

for ds in DATASETS:
    env = dict(RUN_ENV)
    if WITH_SHAPLEY_SWEEP:
        env["C1_WITH_SHAPLEY"] = "1"
    out = RESULTS / f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v6_lambda_sweep"
    rc = stream([sys.executable, str(CODE_DIR / "scripts" / "run_loo_lambda_sweep.py"),
                 "--dataset", ds,
                 "--source-run", str(RESULTS / f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v3_prospective"),
                 "--out", str(out)],
                env=env, log=str(LOGDIR / f"lambda_sweep_{ds}.log"))
    assert rc == 0, f"lambda sweep failed for {ds}"
print("\nDONE. Artifacts: *_v6_lambda_sweep/tables/{lambda_sensitivity_mean_std.csv, validation_tuned_lambda.csv}")


In [ ]:
# Cell 4 — EXPERIMENT 2: SECOND BACKBONE (NGCF) under the identical frozen protocol.
# Only changed factor vs. C1b: propagation scheme (nonlinear W1/W2 + LeakyReLU).
# Amazon first (~2–3 h). ML-1M is long (~12–16 h with Shapley): set RUN_ML1M=True when ready.
RUN_AMAZON = True
RUN_ML1M = False          # flip to True for the long ML-1M run
WITH_SHAPLEY = True       # reviewers need Shapley-vs-LOO on the second backbone

jobs = []
if RUN_AMAZON:
    jobs.append(("amazon", "amazon_books_lightgcn_v3_prospective", "amazon_books_ngcf_v6_second_backbone"))
if RUN_ML1M:
    jobs.append(("ml1m", "ml1m_lightgcn_v3_prospective", "ml1m_ngcf_v6_second_backbone"))

for ds, src, outname in jobs:
    env = dict(RUN_ENV)
    if WITH_SHAPLEY:
        env["C1_WITH_SHAPLEY"] = "1"
    rc = stream([sys.executable, str(CODE_DIR / "scripts" / "run_second_backbone.py"),
                 "--dataset", ds, "--backbone", "ngcf",
                 "--source-run", str(RESULTS / src),
                 "--out", str(RESULTS / outname)],
                env=env, log=str(LOGDIR / f"second_backbone_{ds}.log"))
    assert rc == 0, f"NGCF run failed for {ds}"
print("\nDONE. Artifacts: *_ngcf_v6_second_backbone/raw/per_user_metrics_all.csv.gz + tables/")


In [ ]:
# Cell 5 — EXPERIMENT 3: multi-seed DESIGN ABLATIONS (reviewers: Table 13 is single-seed).
# Seed 42 is already released; this adds more seeds. Output merges into the canonical
# tables/design_ablations.csv with a 'seed' column (per-seed CSVs also kept).
# WARNING: ML-1M is ~10–12 h PER SEED. Start with Amazon + one ML-1M seed.
DATASETS = ["amazon", "ml1m"]
SEEDS = [43]               # extend to [43, 44, 45, 46] as budget allows

for ds in DATASETS:
    for s in SEEDS:
        rc = stream([sys.executable, str(CODE_DIR / "scripts" / "run_design_ablations.py"),
                     "--dataset", ds, "--seed", str(s)],
                    log=str(LOGDIR / f"design_ablations_{ds}_seed{s}.log"))
        assert rc == 0, f"design ablations failed: {ds} seed {s}"
print("\nDONE. Artifact: <v3 run>/tables/design_ablations.csv (multi-seed, 'seed' column)")


In [ ]:
# Cell 6 — EXPERIMENT 4: multi-seed MASKED-FORWARD faithfulness (CPU-forced inside the
# script because MPS gave mask-independent garbage; see script header). 1,000-user subsample.
# ~30–40 min per seed per dataset. Seed 42 already released; adds the rest.
DATASETS = ["ml1m", "amazon"]
SEEDS = [43, 44, 45, 46]
N_USERS = 1000

for ds in DATASETS:
    for s in SEEDS:
        rc = stream([sys.executable, str(CODE_DIR / "scripts" / "run_masked_forward_faithfulness.py"),
                     "--dataset", ds, "--seed", str(s), "--n-users", str(N_USERS)],
                    env=dict(RUN_ENV, COALGAME_DEVICE="cpu"),   # script is CPU-only by design
                    log=str(LOGDIR / f"masked_forward_{ds}_seed{s}.log"))
        assert rc == 0, f"masked-forward failed: {ds} seed {s}"
print("\nDONE. Artifact: <v3 run>/tables/masked_forward_faithfulness.csv (multi-seed)")


In [ ]:
# Cell 7 — EXPERIMENT 5: attribution stability + model randomization + perturbation sanity
# (reviewers R1#11 / R3). Trains seeds 42+43, computes LOO attributions on both plus an
# UNTRAINED model and a noise-perturbed model; reports Spearman / top-12 overlap / rerank NDCG.
DATASETS = ["amazon", "ml1m"]

for ds in DATASETS:
    outname = f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v6_randomization_sanity"
    rc = stream([sys.executable, str(CODE_DIR / "scripts" / "run_randomization_sanity.py"),
                 "--dataset", ds,
                 "--source-run", str(RESULTS / f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v3_prospective"),
                 "--out", str(RESULTS / outname)],
                log=str(LOGDIR / f"randomization_{ds}.log"))
    assert rc == 0, f"randomization sanity failed for {ds}"
print("\nDONE. Artifacts: *_v6_randomization_sanity/randomization_sanity.json")


In [ ]:
# Cell 8 — EXPERIMENT 6: validation-negative-set sensitivity |N^-| ∈ {50, 100, 500}.
# Single training seed (42), diagnostic. Reports attribution Spearman / top-12 overlap vs the
# protocol (100) reference and reranked NDCG@20 for LOO and Shapley at each size.
DATASETS = ["amazon", "ml1m"]
SIZES = [50, 100, 500]

for ds in DATASETS:
    outname = f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v6_negset_sensitivity"
    cmd = [sys.executable, str(CODE_DIR / "scripts" / "run_negset_sensitivity.py"),
           "--dataset", ds,
           "--source-run", str(RESULTS / f"{ds if ds=='ml1m' else 'amazon_books'}_lightgcn_v3_prospective"),
           "--out", str(RESULTS / outname),
           "--sizes"] + [str(x) for x in SIZES]
    rc = stream(cmd, log=str(LOGDIR / f"negset_{ds}.log"))
    assert rc == 0, f"negset sensitivity failed for {ds}"
print("\nDONE. Artifacts: *_v6_negset_sensitivity/negset_sensitivity.json")


In [ ]:
# Cell 9 — Status & artifacts overview (safe to run any time)
import pandas as pd

checks = [
    ("λ-sweep amazon", RESULTS / "amazon_books_lightgcn_v6_lambda_sweep" / "tables" / "validation_tuned_lambda.csv"),
    ("λ-sweep ml1m", RESULTS / "ml1m_lightgcn_v6_lambda_sweep" / "tables" / "validation_tuned_lambda.csv"),
    ("NGCF amazon", RESULTS / "amazon_books_ngcf_v6_second_backbone" / "raw" / "per_user_metrics_all.csv.gz"),
    ("NGCF ml1m", RESULTS / "ml1m_ngcf_v6_second_backbone" / "raw" / "per_user_metrics_all.csv.gz"),
    ("design ablations ml1m (multi-seed)", RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "design_ablations.csv"),
    ("design ablations amazon (multi-seed)", RESULTS / "amazon_books_lightgcn_v3_prospective" / "tables" / "design_ablations.csv"),
    ("masked-forward ml1m", RESULTS / "ml1m_lightgcn_v3_prospective" / "tables" / "masked_forward_faithfulness.csv"),
    ("masked-forward amazon", RESULTS / "amazon_books_lightgcn_v3_prospective" / "tables" / "masked_forward_faithfulness.csv"),
    ("randomization amazon", RESULTS / "amazon_books_lightgcn_v6_randomization_sanity" / "randomization_sanity.json"),
    ("randomization ml1m", RESULTS / "ml1m_lightgcn_v6_randomization_sanity" / "randomization_sanity.json"),
    ("negset amazon", RESULTS / "amazon_books_lightgcn_v6_negset_sensitivity" / "negset_sensitivity.json"),
    ("negset ml1m", RESULTS / "ml1m_lightgcn_v6_negset_sensitivity" / "negset_sensitivity.json"),
]
for name, p in checks:
    mark = "OK " if p.exists() else "-- "
    extra = ""
    if p.exists() and str(p).endswith(".csv") and "ablations" in str(p) or str(p).endswith("masked_forward_faithfulness.csv"):
        try:
            seeds = sorted(pd.read_csv(p)["seed"].unique().tolist()) if p.exists() else []
            extra = f" (seeds {seeds})"
        except Exception:
            pass
    print(f"{mark} {name}{extra}")


In [ ]:
# Cell 10 — Hand back to the loop: commit + push the artifacts
print("Run these from the REPO ROOT (next-paper/):")
print("""
git add paper-ideas/CoalGameRec/code/results/journal_runs paper-ideas/CoalGameRec/code/notebooks/CoalGameRec_R6_Experiments.ipynb
git commit -m "round-6 experiments: LOO lambda sweep, NGCF second backbone, multi-seed ablations/faithfulness, randomization + negset diagnostics"
git push origin arena/019fdd75-next-paper
""")
print("Then paste the outputs (or this notebook's log tails) back to the agent loop.")
